# 03. Phase 2: 自動最適化ループ（自動化のパート）

**役割分担のうち「自動化」のパート。** 人間が到達した 0.8204 を起点に、機械的な探索で精度を伸ばせるか検証する。

## 無限ループ対策（停止条件）
- **(A)** 3 ステップ連続で best が改善しなければ終了
- **(B)** 最大 10 イテレーション
- **(C)** Accuracy >= 0.95 に到達したら終了

施策: ① 特徴選択/多項式特徴 ② GridSearchCV チューニング ③ アンサンブル ④ XGBoost / LightGBM。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '../src')
import pandas as pd
from feature_engineering import build_features_v2, feature_columns
from model_training import build_tuned_models, build_ensembles, grid_search_rf, grid_search_xgb
from evaluation import make_cv, evaluate_models, cv_accuracy

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['Survived']
cv = make_cv(5)

train_v2 = build_features_v2(train, train)
feats2 = feature_columns(train_v2)
X2 = train_v2[feats2]

## 最適化ループ本体
各ステップで CV Accuracy を測定・記録し、停止条件を判定する。

In [ ]:
log = []
BEST = 0.8204            # Phase 1 の到達点を起点に
no_improve = 0
MAX_ITER, TARGET = 10, 0.95

def step(it, desc, model, X):
    global BEST, no_improve
    acc = cv_accuracy(model, X, y, cv)
    improved = acc > BEST + 1e-9
    if improved:
        BEST = acc; no_improve = 0
    else:
        no_improve += 1
    log.append({'iter': it, '施策': desc, 'Accuracy': round(acc,4), 'best': round(BEST,4), '改善': '○' if improved else '×'})
    print(f"[{it}] {desc}: acc={acc:.4f} best={BEST:.4f}{'  ★' if improved else ''}")
    return improved

def should_stop():
    if BEST >= TARGET:
        print('停止(C): 0.95 到達'); return True
    if no_improve >= 3:
        print('停止(A): 3 ステップ連続で改善なし'); return True
    return False

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

it = 0
# Step1: 多項式(交互作用)特徴 + ロジスティック回帰
it += 1
poly_lr = make_pipeline(StandardScaler(), PolynomialFeatures(2, interaction_only=True, include_bias=False),
                        LogisticRegression(max_iter=2000, C=0.5, random_state=42))
step(it, '多項式交互作用 + LR', poly_lr, X2)

# Step2: RandomForest を GridSearch でチューニング
it += 1
rf_g = grid_search_rf(X2, y, cv)
step(it, 'RF GridSearch', rf_g.best_estimator_, X2)
print('   RF best params:', rf_g.best_params_)

# Step3: XGBoost を GridSearch でチューニング
it += 1
xgb_g = grid_search_xgb(X2, y, cv)
step(it, 'XGBoost GridSearch', xgb_g.best_estimator_, X2)
print('   XGB best params:', xgb_g.best_params_)

In [ ]:
# Step4-: チューニング済みモデル群 & アンサンブルを評価
tuned = build_tuned_models()
print('=== チューニング済みモデル ===')
display(evaluate_models(tuned, X2, y, cv))
print('=== アンサンブル ===')
display(evaluate_models(build_ensembles(tuned), X2, y, cv))

## 最適化プロセス記録

In [ ]:
pd.DataFrame(log)

## Phase 2 の結論 — 自動化の「限界」

- **到達した最高精度: 0.8440**（RandomForest / XGBoost が同値で収束）。
- アンサンブル (Voting / Stacking) は 0.836〜0.841 で**個別モデルを超えられず** → モデル間相関が高く多様性不足。
- 停止条件 (A)「3 ステップ連続改善なし」で正常終了。

### 重要な気づき
複数の独立手法が同じ 0.844 に収束したのは、**Titanic データの本質的な説明限界**を示唆する。
生存には運の要素が大きく、特徴量で説明できる上限は概ね 0.83〜0.85。
目標 0.95 は正攻法では到達不可能で、**実務的には『汎化性能の堅実さ』を優先**すべき局面。

最終提出は最高精度かつ頑健な **RandomForest (tuned) 0.8440** を採用 (`data/submission.csv`)。